<a href="https://colab.research.google.com/github/xmessi08/ML-project-6/blob/main/ML_project6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

russian_stopwords = stopwords.words('russian')
stemmer = SnowballStemmer('russian')

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)
    tokens = text.split()
    cleaned_tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in russian_stopwords and len(token) > 2
    ]
    return " ".join(cleaned_tokens)

data = {
    'text': [
        "Отличный сервис, очень быстро доставили заказ, товар качественный!",
        "Ужасный магазин, курьер опоздал на три часа и грубил.",
        "Качество товара просто супер, обязательно буду заказывать еще.",
        "Прислали сломанную вещь, деньги возвращать отказываются!",
        "Нормальный товар за свои деньги, всё пришло вовремя.",
        "Никому не советую этот сервис, полнейший обман и развод.",
        "Прекрасная поддержка, быстро ответили и помогли решить проблему.",
        "Товар шёл два месяца, упакован плохо, коробка мятая.",
        "Замечательный продавец, всё аккуратно упаковано и работает.",
        "Товар вроде нормальный, но доставка ужасно долгая и дорогая."
    ],
    'label': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)
df['cleaned_text'] = df['text'].apply(preprocess_text)

vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=0.9,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['cleaned_text'])
y = df['label']

X_train, X_test, y_train, y_test, text_train, text_test = train_test_split(
    X, y, df['text'], test_size=0.3, random_state=42, stratify=y
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

results_df = pd.DataFrame({
    'text': text_test,
    'true_label': y_test,
    'predicted_label': y_pred,
    'prob_pos': y_probs
})

errors = results_df[results_df['true_label'] != results_df['predicted_label']].head(5)
print("Ошибки модели:")
print(errors)

Ошибки модели:
                                                text  true_label  \
1  Ужасный магазин, курьер опоздал на три часа и ...           0   
7  Товар шёл два месяца, упакован плохо, коробка ...           0   

   predicted_label  prob_pos  
1                1  0.563950  
7                1  0.575938  
